Section A | SQL Querying and Filtering

01. Inspect water_meter_readings_source and display its schema and total row count.
02. Retrieve reading_id, meter_id, zone_id, reading_time and consumption_liters for readings marked Valid.
03. Find readings where pressure_bar is below 2.5, ordering the results from lowest pressure upward.
04. Find records where flow_lpm is between 20 and 50 and the source_system is Gateway or SmartMeter.
05. List distinct reading_status values and count how many readings belong to each status.

In [0]:
%sql
USE CATALOG workspace1;

USE SCHEMA water_operations;

In [0]:
%sql
DESCRIBE water_meter_readings_source;

SELECT COUNT(*) AS total_rows
FROM water_meter_readings_source;

total_rows
157


In [0]:
%sql
SELECT
    reading_id,
    meter_id,
    zone_id,
    reading_time,
    consumption_liters
FROM water_meter_readings_source
WHERE reading_status = 'Valid';

reading_id,meter_id,zone_id,reading_time,consumption_liters
1001,M001,W01,2026-10-01T07:00:00.000Z,2365
1002,M002,W02,2026-10-01T08:10:00.000Z,3545
1003,M003,W03,2026-10-01T09:20:00.000Z,548
1004,M004,W04,2026-10-01T10:00:00.000Z,2871
1006,M006,W06,2026-10-01T12:20:00.000Z,2171
1007,M007,W07,2026-10-01T13:00:00.000Z,4167
1008,M008,W08,2026-10-01T14:10:00.000Z,3621
1009,M009,W09,2026-10-01T15:20:00.000Z,1609
1010,M010,W10,2026-10-01T16:00:00.000Z,1829
1011,M011,W01,2026-10-02T07:10:00.000Z,313


In [0]:
%sql
SELECT *
FROM water_meter_readings_source
WHERE pressure_bar < 2.5
ORDER BY pressure_bar ASC;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1071,M011,W01,2026-10-08T07:10:00.000Z,10.9,1.8,0.8,647,Valid,SmartMeter
1064,M004,W04,2026-10-07T10:00:00.000Z,20.3,1.82,0.84,708,Valid,Gateway
1024,M024,W04,2026-10-03T10:20:00.000Z,44.5,1.83,1.25,2085,Valid,FieldApp
1116,M026,W06,2026-10-12T12:10:00.000Z,28.9,1.84,0.36,1692,Valid,SmartMeter
1133,M013,W03,2026-10-14T09:00:00.000Z,15.6,1.86,0.8,900,Valid,Gateway
1028,M028,W08,2026-10-03T14:00:00.000Z,15.0,1.9,0.94,452,Valid,Gateway
1032,M002,W02,2026-10-04T08:10:00.000Z,34.4,1.92,0.88,1006,Valid,SmartMeter
1051,M021,W01,2026-10-06T07:20:00.000Z,72.0,1.93,0.27,1818,Valid,FieldApp
1052,M022,W02,2026-10-06T08:00:00.000Z,74.0,1.94,0.21,4669,Valid,Gateway
1044,M014,W04,2026-10-05T10:10:00.000Z,79.3,1.95,1.23,2661,Valid,SmartMeter


In [0]:
%sql
SELECT *
FROM water_meter_readings_source
WHERE flow_lpm BETWEEN 20 AND 50
AND source_system IN ('Gateway', 'SmartMeter');

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway
1014,M014,W04,2026-10-02T10:10:00.000Z,22.4,4.73,0.5,1108,Valid,SmartMeter
1017,M017,W07,2026-10-02T13:10:00.000Z,28.0,3.77,1.08,1318,Valid,SmartMeter
1019,M019,W09,2026-10-02T15:00:00.000Z,23.0,4.34,0.28,1091,Valid,Gateway
1022,M022,W02,2026-10-03T08:00:00.000Z,32.5,3.35,0.6,1706,Suspect,Gateway
1032,M002,W02,2026-10-04T08:10:00.000Z,34.4,1.92,0.88,1006,Valid,SmartMeter
1047,M017,W07,2026-10-05T13:10:00.000Z,28.8,3.23,0.91,1258,Valid,SmartMeter
1049,M019,W09,2026-10-05T15:00:00.000Z,21.4,2.43,0.47,611,Valid,Gateway
1050,M020,W10,2026-10-05T16:10:00.000Z,37.2,3.91,1.02,2154,Valid,SmartMeter
1056,M026,W06,2026-10-06T12:10:00.000Z,27.2,2.69,0.68,1131,Suspect,SmartMeter


In [0]:
%sql
SELECT
    reading_status,
    COUNT(*) AS reading_count
FROM water_meter_readings_source
GROUP BY reading_status
ORDER BY reading_count DESC;

reading_status,reading_count
Valid,137
Suspect,11
Rejected,9


Section B | Joins, Aggregations and Window Functions

06. Join meter readings with water_zones and show zone_name, district, meter_id and consumption_liters.
07. Find reading records whose zone_id has no matching entry in water_zones.
08. Find zones that currently have no meter readings, using a suitable join.
09. Calculate total valid consumption, average pressure and reading count for each zone; exclude
nonpositive consumption.
10. Rank meters by total valid consumption within each district using a SQL window function

In [0]:
%sql
SELECT
    z.zone_name,
    z.district,
    r.meter_id,
    r.consumption_liters
FROM water_meter_readings_source r
INNER JOIN water_zones z
ON r.zone_id = z.zone_id;

zone_name,district,meter_id,consumption_liters
Old City,Central,M001,2365
Hill View,North,M002,3545
Market District,Central,M003,548
Lake Side,West,M004,2871
University Quarter,East,M005,2656
Industrial Estate,South,M006,2171
Garden Colony,North,M007,4167
Transit Hub,East,M008,3621
Riverfront,West,M009,1609
South Extension,South,M010,1829


In [0]:
%sql
SELECT r.*
FROM water_meter_readings_source r
LEFT JOIN water_zones z
ON r.zone_id = z.zone_id
WHERE z.zone_id IS NULL;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1154,M999,W99,2026-10-16T11:00:00.000Z,24.5,3.0,0.45,1150,Valid,FieldApp
1156,M012,null,2026-10-16T12:30:00.000Z,28.0,3.1,0.52,870,Valid,Gateway


In [0]:
%sql
SELECT
    z.zone_id,
    z.zone_name,
    z.district
FROM water_zones z
LEFT JOIN water_meter_readings_source r
ON z.zone_id = r.zone_id
WHERE r.zone_id IS NULL;

zone_id,zone_name,district
W11,New Development,West


In [0]:
%sql
SELECT
    zone_id,
    SUM(consumption_liters) AS total_consumption,
    AVG(pressure_bar) AS average_pressure,
    COUNT(*) AS reading_count
FROM water_meter_readings_source
WHERE reading_status = 'Valid'
AND consumption_liters > 0
GROUP BY zone_id
ORDER BY total_consumption DESC;

zone_id,total_consumption,average_pressure,reading_count
W03,35665,3.019999999999999,13
W08,33689,2.8657142857142857,14
W05,28717,3.632857142857143,14
W10,27808,3.717692307692307,14
W07,25104,3.5700000000000003,13
W02,24120,3.3061538461538458,13
W01,23107,3.423076923076923,13
W09,20894,3.5469230769230773,13
W04,20781,3.2371428571428575,14
W06,20212,3.160769230769231,13


In [0]:
%sql
WITH meter_totals AS (
    SELECT
        z.district,
        r.meter_id,
        SUM(r.consumption_liters) AS total_consumption
    FROM water_meter_readings_source r
    INNER JOIN water_zones z
    ON r.zone_id = z.zone_id
    WHERE r.reading_status = 'Valid'
    AND r.consumption_liters > 0
    GROUP BY z.district, r.meter_id
)

SELECT
    district,
    meter_id,
    total_consumption,
    RANK() OVER (
        PARTITION BY district
        ORDER BY total_consumption DESC
    ) AS meter_rank
FROM meter_totals
ORDER BY district, meter_rank;

district,meter_id,total_consumption,meter_rank
Central,M023,17814,1
Central,M001,11416,2
Central,M003,10447,3
Central,M013,7404,4
Central,M021,7395,5
Central,M011,4296,6
East,M008,14095,1
East,M018,10660,2
East,M025,10536,3
East,M015,9807,4


Section E | Delta Lake Tables and CRUD

21. Create water_readings_work_copy using CREATE TABLE AS SELECT from water_meter_readings_work.
22. Insert two new readings into water_readings_work_copy using SQL; use new unique reading IDs.
23. Update the status of a chosen Suspect reading in the copy to Rejected and verify the change.
24. Delete Rejected readings from the copy and compare counts before and after deletion.
Page 2
Azure Databricks  |  City Water Network Monitoring
25. Use PySpark to create a Delta table named zone_consumption_summary from an aggregated
DataFrame.

In [0]:
%sql
CREATE OR REPLACE TABLE water_readings_work_copy
AS
SELECT *
FROM water_meter_readings_work;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM water_readings_work_copy;

total_rows
157


In [0]:
%sql
SELECT MAX(reading_id) AS max_reading_id
FROM water_readings_work_copy;

max_reading_id
1156


In [0]:
%sql
INSERT INTO water_readings_work_copy
VALUES
(158, 'MTR_NEW01', 'Z001', current_timestamp(), 35.5, 3.2, 0.5, 120, 'Valid', 'SmartMeter'),
(159, 'MTR_NEW02', 'Z002', current_timestamp(), 42.0, 2.8, 0.4, 150, 'Valid', 'Gateway');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
SELECT *
FROM water_readings_work_copy
WHERE reading_id IN (158, 159);

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
158,MTR_NEW01,Z001,2026-11-11T15:03:52.299Z,35.5,3.2,0.5,120,Valid,SmartMeter
159,MTR_NEW02,Z002,2026-11-11T15:03:52.299Z,42.0,2.8,0.4,150,Valid,Gateway


In [0]:
%sql
SELECT reading_id, meter_id, reading_status
FROM water_readings_work_copy
WHERE reading_status = 'Suspect'
LIMIT 5;

reading_id,meter_id,reading_status
1005,M005,Suspect
1022,M022,Suspect
1039,M009,Suspect
1056,M026,Suspect
1073,M013,Suspect


In [0]:
%sql
UPDATE water_readings_work_copy
SET reading_status = 'Rejected'
WHERE reading_id = 1022;

num_affected_rows
1


In [0]:
%sql
SELECT reading_id, reading_status
FROM water_readings_work_copy
WHERE reading_id = 1022;

reading_id,reading_status
1022,Rejected


In [0]:
%sql
SELECT COUNT(*) AS rejected_before
FROM water_readings_work_copy
WHERE reading_status = 'Rejected';

rejected_before
10


In [0]:
%sql
DELETE FROM water_readings_work_copy
WHERE reading_status = 'Rejected';

num_affected_rows
10


In [0]:
%sql
SELECT COUNT(*) AS rejected_after
FROM water_readings_work_copy
WHERE reading_status = 'Rejected';

rejected_after
0


In [0]:
%python
from pyspark.sql import functions as F

clean_df = spark.table("water_readings_clean")
zones_df = spark.table("water_zones")

zone_summary_df = clean_df.groupBy("zone_id").agg(
    F.sum("consumption_liters").alias("total_consumption"),
    F.avg("consumption_liters").alias("average_consumption"),
    F.count("*").alias("reading_count")
)

zone_summary_df = zone_summary_df.join(
    zones_df.select("zone_id", "zone_name", "district"),
    on="zone_id",
    how="left"
)

zone_summary_df = zone_summary_df.select(
    "zone_id",
    "zone_name",
    "district",
    "total_consumption",
    "average_consumption",
    "reading_count"
)

zone_summary_df.write.format("delta").mode("overwrite").saveAsTable(
    "zone_consumption_summary"
)

In [0]:
%python
summary_df = spark.table("zone_consumption_summary")
summary_df.show(truncate=False)

+-------+------------------+--------+-----------------+-------------------+-------------+
|zone_id|zone_name         |district|total_consumption|average_consumption|reading_count|
+-------+------------------+--------+-----------------+-------------------+-------------+
|W06    |Industrial Estate |South   |20212            |1554.7692307692307 |13           |
|W10    |South Extension   |South   |27808            |1986.2857142857142 |14           |
|W07    |Garden Colony     |North   |25104            |1931.076923076923  |13           |
|W08    |Transit Hub       |East    |33689            |2406.3571428571427 |14           |
|W02    |Hill View         |North   |24120            |1855.3846153846155 |13           |
|W03    |Market District   |Central |35665            |2743.4615384615386 |13           |
|W99    |NULL              |NULL    |1150             |1150.0             |1            |
|W04    |Lake Side         |West    |19481            |1498.5384615384614 |13           |
|W01    |O

Section F | MERGE, UPSERT and SCD

26. Compare water_reading_changes with water_meter_readings_work and separate matching IDs from
new IDs.
27. Run a SQL MERGE on water_meter_readings_work using reading_id as the key: update matches and
insert nonmatches.
28. Validate that changed reading IDs have new values and that new IDs were inserted.
29. Perform the same SCD Type 1 MERGE using the PySpark DeltaTable API on a fresh working-table copy.
30. Build a zone_capacity_history SCD Type 2 table from water_zones. Effective 2026-10-20, change W02
capacity to 195000 and W06 to 305000, retaining old versions and current flags

In [0]:
USE CATALOG workspace1;
USE SCHEMA water_operations;

In [0]:
DESCRIBE water_reading_changes;
DESCRIBE water_meter_readings_work;

col_name,data_type,comment
reading_id,int,null
meter_id,string,null
zone_id,string,null
reading_time,timestamp,null
flow_lpm,double,null
pressure_bar,double,null
chlorine_mg_l,double,null
consumption_liters,int,null
reading_status,string,null
source_system,string,null


In [0]:
SELECT c.*
FROM water_reading_changes c
INNER JOIN water_meter_readings_work w
ON c.reading_id = w.reading_id;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1005,M005,W05,2026-10-01T11:10:00.000Z,64.3,3.24,0.43,2776,Valid,FieldApp
1027,M027,W07,2026-10-03T13:20:00.000Z,54.6,4.99,0.95,2678,Valid,FieldApp
1062,M002,W02,2026-10-07T08:10:00.000Z,30.1,2.43,0.3,1057,Valid,FieldApp
1088,M028,W08,2026-10-09T14:00:00.000Z,81.4,2.17,1.06,4931,Valid,FieldApp
1119,M029,W09,2026-10-12T15:10:00.000Z,22.1,3.63,1.13,1339,Valid,FieldApp
1145,M025,W05,2026-10-15T11:00:00.000Z,77.5,3.96,1.29,4908,Valid,FieldApp


In [0]:
SELECT c.*
FROM water_reading_changes c
LEFT ANTI JOIN water_meter_readings_work w
ON c.reading_id = w.reading_id;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1201,M002,W02,2026-10-18T08:00:00.000Z,24.0,2.6,0.45,1100,Valid,FieldApp
1202,M005,W05,2026-10-18T09:00:00.000Z,28.7,2.7,0.5,1250,Valid,FieldApp
1203,M008,W08,2026-10-18T10:00:00.000Z,33.4,2.8,0.55,1400,Valid,FieldApp
1204,M011,W01,2026-10-18T11:00:00.000Z,38.1,2.9,0.6,1550,Valid,FieldApp
1205,M014,W04,2026-10-18T12:00:00.000Z,42.8,3.0,0.65,1700,Valid,FieldApp
1206,M017,W07,2026-10-18T13:00:00.000Z,47.5,3.1,0.7,1850,Valid,FieldApp


In [0]:
MERGE INTO water_meter_readings_work AS target
USING water_reading_changes AS source
ON target.reading_id = source.reading_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
12,6,0,6


In [0]:
SELECT w.*
FROM water_meter_readings_work w
INNER JOIN water_reading_changes c
ON w.reading_id = c.reading_id
ORDER BY w.reading_id;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1005,M005,W05,2026-10-01T11:10:00.000Z,64.3,3.24,0.43,2776,Valid,FieldApp
1027,M027,W07,2026-10-03T13:20:00.000Z,54.6,4.99,0.95,2678,Valid,FieldApp
1062,M002,W02,2026-10-07T08:10:00.000Z,30.1,2.43,0.3,1057,Valid,FieldApp
1088,M028,W08,2026-10-09T14:00:00.000Z,81.4,2.17,1.06,4931,Valid,FieldApp
1119,M029,W09,2026-10-12T15:10:00.000Z,22.1,3.63,1.13,1339,Valid,FieldApp
1145,M025,W05,2026-10-15T11:00:00.000Z,77.5,3.96,1.29,4908,Valid,FieldApp
1201,M002,W02,2026-10-18T08:00:00.000Z,24.0,2.6,0.45,1100,Valid,FieldApp
1202,M005,W05,2026-10-18T09:00:00.000Z,28.7,2.7,0.5,1250,Valid,FieldApp
1203,M008,W08,2026-10-18T10:00:00.000Z,33.4,2.8,0.55,1400,Valid,FieldApp
1204,M011,W01,2026-10-18T11:00:00.000Z,38.1,2.9,0.6,1550,Valid,FieldApp


In [0]:
SELECT COUNT(*) AS missing_ids
FROM water_reading_changes c
LEFT ANTI JOIN water_meter_readings_work w
ON c.reading_id = w.reading_id;

missing_ids
0


In [0]:
%python
spark.sql("""
CREATE OR REPLACE TABLE water_readings_pyspark_copy
AS
SELECT * FROM water_meter_readings_source
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
%python
from delta.tables import DeltaTable

changes_df = spark.table("water_reading_changes")

target_table = DeltaTable.forName(
    spark,
    "water_readings_pyspark_copy"
)

In [0]:
%python
target_table.alias("target").merge(
    changes_df.alias("source"),
    "target.reading_id = source.reading_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%python
updated_df = spark.table("water_readings_pyspark_copy")

print("Total rows:", updated_df.count())
updated_df.orderBy("reading_id").show(truncate=False)

Total rows: 163
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|reading_id|meter_id|zone_id|reading_time       |flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|1001      |M001    |W01    |2026-10-01 07:00:00|39.1    |3.56        |0.75         |2365              |Valid         |Gateway      |
|1002      |M002    |W02    |2026-10-01 08:10:00|59.9    |4.6         |0.39         |3545              |Valid         |SmartMeter   |
|1003      |M003    |W03    |2026-10-01 09:20:00|25.7    |2.22        |1.15         |548               |Valid         |FieldApp     |
|1004      |M004    |W04    |2026-10-01 10:00:00|56.7    |2.84        |1.1          |2871              |Valid         |Gateway      |
|1005      |M005    |W05    |2026-10-01 11:10:

In [0]:
CREATE OR REPLACE TABLE zone_capacity_history
AS


SELECT
    zone_id,
    zone_name,
    city,
    district,
    zone_type,
    daily_capacity_liters,
    active,
    commissioned_on,
    DATE '1900-01-01' AS effective_from,
    CASE
        WHEN zone_id IN ('W02', 'W06')
        THEN DATE '2026-10-19'
        ELSE DATE '9999-12-31'
    END AS effective_to,
    CASE
        WHEN zone_id IN ('W02', 'W06') THEN false
        ELSE true
    END AS is_current
FROM water_zones

UNION ALL


SELECT
    zone_id,
    zone_name,
    city,
    district,
    zone_type,
    195000 AS daily_capacity_liters,
    active,
    commissioned_on,
    DATE '2026-10-20' AS effective_from,
    DATE '9999-12-31' AS effective_to,
    true AS is_current
FROM water_zones
WHERE zone_id = 'W02'

UNION ALL


SELECT
    zone_id,
    zone_name,
    city,
    district,
    zone_type,
    305000 AS daily_capacity_liters,
    active,
    commissioned_on,
    DATE '2026-10-20' AS effective_from,
    DATE '9999-12-31' AS effective_to,
    true AS is_current
FROM water_zones
WHERE zone_id = 'W06';

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM zone_capacity_history
WHERE zone_id IN ('W02', 'W06')
ORDER BY zone_id, effective_from;

zone_id,zone_name,city,district,zone_type,daily_capacity_liters,active,commissioned_on,effective_from,effective_to,is_current
W02,Hill View,Hyderabad,North,Residential,180000,true,2019-02-15,1900-01-01,2026-10-19,false
W02,Hill View,Hyderabad,North,Residential,195000,true,2019-02-15,2026-10-20,9999-12-31,true
W06,Industrial Estate,Hyderabad,South,Industrial,320000,true,2023-06-15,1900-01-01,2026-10-19,false
W06,Industrial Estate,Hyderabad,South,Industrial,305000,true,2023-06-15,2026-10-20,9999-12-31,true


In [0]:
SELECT zone_id, zone_name, daily_capacity_liters,
       effective_from, effective_to, is_current
FROM zone_capacity_history
WHERE is_current = true
ORDER BY zone_id;

zone_id,zone_name,daily_capacity_liters,effective_from,effective_to,is_current
W01,Old City,240000,1900-01-01,9999-12-31,true
W02,Hill View,195000,2026-10-20,9999-12-31,true
W03,Market District,210000,1900-01-01,9999-12-31,true
W04,Lake Side,280000,1900-01-01,9999-12-31,true
W05,University Quarter,150000,1900-01-01,9999-12-31,true
W06,Industrial Estate,305000,2026-10-20,9999-12-31,true
W07,Garden Colony,190000,1900-01-01,9999-12-31,true
W08,Transit Hub,170000,1900-01-01,9999-12-31,true
W09,Riverfront,230000,1900-01-01,9999-12-31,true
W10,South Extension,260000,1900-01-01,9999-12-31,true


Section G | Delta History, Time Travel and Maintenance

31. Use DESCRIBE DETAIL on water_meter_readings_work to inspect the format, storage location, file count
and size.
32. Use DESCRIBE HISTORY to identify recorded table operations and locate the version produced by
MERGE.
33. Query the version immediately before the MERGE using VERSION AS OF; compare reading 1005 with its
current value.
34. Execute OPTIMIZE on a suitable working Delta table; compare its file count before and after and explain
the result.
35. Explain the purpose of VACUUM and inspect eligible files with a dry run where supported; do not delete
retained historical files

In [0]:
USE CATALOG workspace1;
USE SCHEMA water_operations;

In [0]:
DESCRIBE DETAIL water_meter_readings_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,f6bf3cfb-9bbe-43e9-8a6d-82314f5b1587,workspace1.water_operations.water_meter_readings_work,null,,2026-11-11T14:48:06.817Z,2026-11-11T15:29:37.000Z,List(),List(),2,9592,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 7db9d29c-f3bd-4254-a174-836b38f0ae2f, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-28df69df-409a-4e97-a378-344c71e234ae, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-5c4f9120-ff1e-42ae-bb04-d14e41ffa6b6)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 6, numDeletionVectors -> 1)",true


In [0]:
DESCRIBE HISTORY water_meter_readings_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-11-11T15:29:37.000Z,74600853328676,luxya2026@gmail.com,MERGE,"Map(predicate -> [""(reading_id#22426 = reading_id#22446)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(3533282780835056),1b002686-893d-4072-aab5-d1da4c101866,1111-144229-v9alz234-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4168, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 6, executionTimeMs -> 2927, materializeSourceTimeMs -> 6, numTargetRowsInserted -> 6, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1049, numTargetRowsUpdated -> 6, numOutputRows -> 12, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 12, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1807)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-11T14:48:09.000Z,74600853328676,luxya2026@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""7db9d29c-f3bd-4254-a174-836b38f0ae2f"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-28df69df-409a-4e97-a378-344c71e234ae"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-5c4f9120-ff1e-42ae-bb04-d14e41ffa6b6""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(3533282780835055),64ddc79d-b78a-4c57-bcef-09d12564acf6,1111-144229-v9alz234-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 157, numOutputBytes -> 5424)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


In [0]:
DESCRIBE HISTORY water_meter_readings_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-11-11T15:29:37.000Z,74600853328676,luxya2026@gmail.com,MERGE,"Map(predicate -> [""(reading_id#22426 = reading_id#22446)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(3533282780835056),1b002686-893d-4072-aab5-d1da4c101866,1111-144229-v9alz234-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4168, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 6, executionTimeMs -> 2927, materializeSourceTimeMs -> 6, numTargetRowsInserted -> 6, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1049, numTargetRowsUpdated -> 6, numOutputRows -> 12, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 12, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1807)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-11T14:48:09.000Z,74600853328676,luxya2026@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""7db9d29c-f3bd-4254-a174-836b38f0ae2f"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-28df69df-409a-4e97-a378-344c71e234ae"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-5c4f9120-ff1e-42ae-bb04-d14e41ffa6b6""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(3533282780835055),64ddc79d-b78a-4c57-bcef-09d12564acf6,1111-144229-v9alz234-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 157, numOutputBytes -> 5424)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


In [0]:
SELECT reading_id, meter_id, consumption_liters, reading_status
FROM water_meter_readings_work VERSION AS OF 0
WHERE reading_id = 1005;

reading_id,meter_id,consumption_liters,reading_status
1005,M005,2656,Suspect


In [0]:
SELECT reading_id, meter_id, consumption_liters, reading_status
FROM water_meter_readings_work
WHERE reading_id = 1005;

reading_id,meter_id,consumption_liters,reading_status
1005,M005,2776,Valid


In [0]:
DESCRIBE DETAIL water_meter_readings_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,f6bf3cfb-9bbe-43e9-8a6d-82314f5b1587,workspace1.water_operations.water_meter_readings_work,null,,2026-11-11T14:48:06.817Z,2026-11-11T15:29:37.000Z,List(),List(),2,9592,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 7db9d29c-f3bd-4254-a174-836b38f0ae2f, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-28df69df-409a-4e97-a378-344c71e234ae, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-5c4f9120-ff1e-42ae-bb04-d14e41ffa6b6)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 6, numDeletionVectors -> 1)",true


In [0]:
OPTIMIZE water_meter_readings_work;

path,metrics
,"List(1, 2, List(6709, 6709, 6709.0, 1, 6709), List(4168, 5424, 4796.0, 2, 9592), 0, null, null, 0, 1, 2, 0, true, 0, 0, 1794411369737, 1794411372246, 8, 1, null, List(1, 6), null, 10, 10, 405, 0, null, null, 0)"


In [0]:
DESCRIBE DETAIL water_meter_readings_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,f6bf3cfb-9bbe-43e9-8a6d-82314f5b1587,workspace1.water_operations.water_meter_readings_work,null,,2026-11-11T14:48:06.817Z,2026-11-11T15:36:12.000Z,List(),List(),1,6709,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 7db9d29c-f3bd-4254-a174-836b38f0ae2f, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-28df69df-409a-4e97-a378-344c71e234ae, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-5c4f9120-ff1e-42ae-bb04-d14e41ffa6b6)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


In [0]:
VACUUM water_meter_readings_work DRY RUN;

path


Section H | Unity Catalog, Views and Parquet

36. Identify the catalog and schema containing your assessment tables; display the managed Delta table
location.
37. Create a permanent view for valid readings with zone names and districts.
38. Create a session-scoped temporary view for low-pressure readings and query it.
39. Create and query a global temporary view of High-priority service requests if your compute supports it;
explain its scope.


In [0]:
SHOW CATALOGS;

catalog
samples
system
workspace
workspace1


In [0]:
SHOW SCHEMAS IN workspace1;

databaseName
default
information_schema
water_operations


In [0]:
SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
workspace1,water_operations


In [0]:
DESCRIBE DETAIL water_meter_readings_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,f6bf3cfb-9bbe-43e9-8a6d-82314f5b1587,workspace1.water_operations.water_meter_readings_work,null,,2026-11-11T14:48:06.817Z,2026-11-11T15:36:12.000Z,List(),List(),1,6709,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 7db9d29c-f3bd-4254-a174-836b38f0ae2f, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-28df69df-409a-4e97-a378-344c71e234ae, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-5c4f9120-ff1e-42ae-bb04-d14e41ffa6b6)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


In [0]:
CREATE OR REPLACE VIEW valid_readings_view AS
SELECT
    r.reading_id,
    r.meter_id,
    r.zone_id,
    z.zone_name,
    z.district,
    r.reading_time,
    r.consumption_liters,
    r.pressure_bar,
    r.reading_status
FROM water_meter_readings_work r
INNER JOIN water_zones z
ON r.zone_id = z.zone_id
WHERE r.reading_status = 'Valid';

In [0]:
SELECT *
FROM valid_readings_view;

reading_id,meter_id,zone_id,zone_name,district,reading_time,consumption_liters,pressure_bar,reading_status
1001,M001,W01,Old City,Central,2026-10-01T07:00:00.000Z,2365,3.56,Valid
1002,M002,W02,Hill View,North,2026-10-01T08:10:00.000Z,3545,4.6,Valid
1003,M003,W03,Market District,Central,2026-10-01T09:20:00.000Z,548,2.22,Valid
1004,M004,W04,Lake Side,West,2026-10-01T10:00:00.000Z,2871,2.84,Valid
1006,M006,W06,Industrial Estate,South,2026-10-01T12:20:00.000Z,2171,2.52,Valid
1007,M007,W07,Garden Colony,North,2026-10-01T13:00:00.000Z,4167,3.19,Valid
1008,M008,W08,Transit Hub,East,2026-10-01T14:10:00.000Z,3621,3.39,Valid
1009,M009,W09,Riverfront,West,2026-10-01T15:20:00.000Z,1609,4.57,Valid
1010,M010,W10,South Extension,South,2026-10-01T16:00:00.000Z,1829,2.38,Valid
1011,M011,W01,Old City,Central,2026-10-02T07:10:00.000Z,313,4.39,Valid


In [0]:
CREATE OR REPLACE TEMP VIEW low_pressure_readings_view AS
SELECT *
FROM water_meter_readings_work
WHERE pressure_bar < 2.5;

In [0]:
SELECT *
FROM low_pressure_readings_view;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway
1016,M016,W06,2026-10-02T12:00:00.000Z,12.9,2.26,0.72,613,Valid,Gateway
1018,M018,W08,2026-10-02T14:20:00.000Z,70.5,2.09,0.3,3709,Valid,FieldApp
1024,M024,W04,2026-10-03T10:20:00.000Z,44.5,1.83,1.25,2085,Valid,FieldApp
1028,M028,W08,2026-10-03T14:00:00.000Z,15.0,1.9,0.94,452,Valid,Gateway
1031,M001,W01,2026-10-04T07:00:00.000Z,80.1,2.38,1.21,3504,Valid,Gateway
1032,M002,W02,2026-10-04T08:10:00.000Z,34.4,1.92,0.88,1006,Valid,SmartMeter
1033,M003,W03,2026-10-04T09:20:00.000Z,56.6,2.19,1.24,2184,Valid,FieldApp
1036,M006,W06,2026-10-04T12:20:00.000Z,26.2,2.09,0.56,1065,Valid,FieldApp


In [0]:
CREATE OR REPLACE GLOBAL TEMP VIEW high_priority_requests_view AS
SELECT *
FROM water_service_requests
WHERE priority = 'High';

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8624028955930819>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "CREATE OR REPLACE GLOBAL TEMP VIEW high_priority_requests_view AS\nSELECT *\nFROM water_service_requests\nWHERE priority = 'High';\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:217, in SqlMagic.sql(

In [0]:
SELECT *
FROM global_temp.high_priority_requests_view;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8624028955930820>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT *\nFROM global_temp.high_priority_requests_view;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:217, in SqlMagic.sql(self, line, cell)
    210 except BaseException as e:
    211     self.drive

In [0]:
SELECT *
FROM workspace1.water_operations.zone_consumption_gold_pipeline;